In [10]:
import pandas as pd
import numpy as np
from DS_processing import Process_train_DS, Process_test_DS

In [11]:
train_path = "/Users/matteogiardina/Desktop/BEMACS 2/2nd semester/ML Project/Data/train.csv"
training_raw = pd.read_csv(train_path)
training_raw = training_raw.drop(columns=["Unnamed: 0"])
print(len(training_raw))

110930


In [12]:
train_df = Process_train_DS(training_raw)

Dropped 0 rows because their 24-hour window hasn't expired yet.


In [13]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 110930 entries, 0 to 110929
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype
---  ------                  --------------   -----
 0   Agency                  110930 non-null  str  
 1   Incident Zip            110930 non-null  str  
 2   Police Precinct         110930 non-null  str  
 3   Borough                 110930 non-null  str  
 4   Open Data Channel Type  110930 non-null  str  
 5   Problem_Grouped         110930 non-null  str  
 6   Location_Grouped        110930 non-null  str  
 7   Is_Landmark             110930 non-null  int64
 8   Is_Taxi                 110930 non-null  int64
 9   Created_Hour            110930 non-null  int32
 10  Created_DayOfWeek       110930 non-null  int32
 11  Created_Month           110930 non-null  int32
 12  Is_Weekend              110930 non-null  int64
 13  Y                       110930 non-null  int64
dtypes: int32(3), int64(4), str(7)
memory usage: 17.0 MB


In [14]:
test_path = "/Users/matteogiardina/Desktop/BEMACS 2/2nd semester/ML Project/Data/test.csv"
test_raw = pd.read_csv(test_path)
test_raw = test_raw.drop(columns=["Unnamed: 0"])
print(len(test_raw))

27733


In [15]:
test_df = Process_test_DS(test_raw)

In [16]:
test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 27733 entries, 0 to 27732
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Agency                  27733 non-null  str  
 1   Incident Zip            27733 non-null  str  
 2   Police Precinct         27733 non-null  str  
 3   Borough                 27733 non-null  str  
 4   Open Data Channel Type  27733 non-null  str  
 5   Problem_Grouped         27733 non-null  str  
 6   Location_Grouped        27733 non-null  str  
 7   Is_Landmark             27733 non-null  int64
 8   Is_Taxi                 27733 non-null  int64
 9   Created_Hour            27733 non-null  int32
 10  Created_DayOfWeek       27733 non-null  int32
 11  Created_Month           27733 non-null  int32
 12  Is_Weekend              27733 non-null  int64
dtypes: int32(3), int64(3), str(7)
memory usage: 4.0 MB


In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier 
import category_encoders as ce

# 1. Grab 100% of your training data (No more train_test_split!)
X = train_df.drop(columns=['Y'])
y = train_df['Y'].astype(int) 

# 2. Your blind TEST set
X_test_blind = test_df.copy() 

# 3. Define categorical columns
cols_to_encode = [
    'Agency', 'Incident Zip', 'Police Precinct', 'Borough', 
    'Open Data Channel Type', 'Problem_Grouped', 'Location_Grouped'
]

# 4. Build the Pipeline
rf_pipeline = Pipeline(steps=[
    ('encoder', ce.TargetEncoder(cols=cols_to_encode, smoothing=10)),
    ('model', RandomForestClassifier(
        n_estimators=100,      
        max_depth=15,          
        class_weight='balanced', 
        random_state=42,       
        n_jobs=-1              
    ))
])

# ==========================================
# 5. THE INTERNAL GRADE: 5-Fold Cross Validation (ACCURACY MODE)
# ==========================================
print("Running 5-Fold Cross Validation for ACCURACY...")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# CHANGED: scoring='accuracy'
cv_scores = cross_val_score(rf_pipeline, X, y, cv=cv, scoring='accuracy', n_jobs=-1)

print("-" * 40)
print(f"Accuracy Scores for each fold: {np.round(cv_scores, 3)}")
print(f"TRUE AVERAGE ACCURACY: {np.mean(cv_scores):.2%} (+/- {np.std(cv_scores):.2%})")
print("-" * 40)

Running 5-Fold Cross Validation for ACCURACY...
----------------------------------------
Accuracy Scores for each fold: [0.87  0.876 0.87  0.868 0.874]
TRUE AVERAGE ACCURACY: 87.17% (+/- 0.28%)
----------------------------------------
